<h1>NAO Teleconnections - Precip - UFS vs SEAS5</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [1]:
basedir = f'../../../..'

In [2]:
import os
import sys
import gc
import collections
import numpy as np
from io import BytesIO
from PIL import Image
import scipy
import xarray as xr
import matplotlib.pyplot as plt

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

from src.datareader import datareader as dr
from src.datareader import UFS_DataReader, ERA5_DataReader
from src.regridder import Regrid
from src.util import oni, regridutil, stats

import warnings
warnings.filterwarnings('ignore')

<h3>User Configurables</h3>

In [3]:
# seas5 sst data are here:
seas5_dir = '/groups/ORC-CLIMATE/aoes_repo/models/seas5/monthly/mean/mslp/'

In [4]:
# # seas5 sst data are here:
# seas5_dir = '/groups/ORC-CLIMATE/aoes_repo/models/seas5/monthly/mean/precip/'

In [5]:
ufs_experiment = 'beta.0.1'

In [6]:
ufs_var = 'prmsl'
seas5_var = 'msl'

In [7]:
# ufs_var = 'pratesfc'
# seas5_var = 'tprate'

In [8]:
# ufs_var = 'pratesfc'
# seas5_var = 'tprate'

In [9]:
time_range = ("1994-01-01", "2020-12-01")
initmonth = 5

# Form Composites around these leads.
# You could also specify a single lead, like, leads=1
leads = 3

In [10]:
# Scale variables for unit-matching
ufs_scaling_factor = 86400
seas5_scaling_factor = 86400000

In [11]:
# For NAO, there are 2 reference locations:
region_1 = {'latmin': 65.0, 'lonmin': 331.2}
region_2 = {'latmin': 37.7, 'lonmin': 334.3}

In [12]:
# Filter ONI events per specific criteria.
strength = None  # For example, you can set strength = 'Weak', 'Moderate', 'Strong', or 'Very Strong'
oni_threshold = None  # For example, you can set oni_threshold = 3

In [13]:
# Transform leads into a tuple used for slicing.
if isinstance(leads, int):
    leads = tuple([leads])

<h2>First step is to calculate NAO Index</h2>

<h5>Get DataReaders</h5>

In [25]:
ufs_data_reader = dr.getDataReader(datasource='UFS',           
                                   experiment = ufs_experiment,
                                   # filename=f'experiments/phase_1/{ufs_model_1}/atm_monthly.zarr',                     
                                   model='atm')

No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/beta.0.1/atm_monthly.zarr


In [28]:
# Collect all sst seas5 files in known directory.
seas5_file_list = os.listdir(seas5_dir)

# Prepend file paths.
seas5_file_list = [os.path.join(seas5_dir, this_file) for this_file in seas5_file_list]

In [29]:
# netcdf4 package is needed here.
seas5_ds = xr.open_mfdataset(seas5_file_list, engine='netcdf4')

In [30]:
# Rename dimensions/coordinates to match our init+lead paradigm.
seas5_ds = seas5_ds.rename_dims({'forecast_reference_time': 'init',
                                 'forecastMonth': 'lead',
                                  'number': 'member'})

seas5_ds = seas5_ds.rename({'forecast_reference_time': 'init',
                            'forecastMonth': 'lead',
                            'number': 'member'})

# By default the lead unit is defined as '1', but we know it's really 'months'.
seas5_ds['lead'].attrs['units'] = 'months'

# Wrap it up into a DataReader object.
seas5_data_reader = dr.getDataReader(datasource='SUPPLIED', dataset=seas5_ds)

In [35]:
seas5_ds_1 = seas5_data_reader.retrieve(var=seas5_var,
                                        lat=region_1['latmin'],
                                        lon=region_1['lonmin'],
                                        initmonths=initmonth,
                                        time=time_range,
                                        member=(0, 24), # We know that seas5 hindcast has members 0-24.
                                        ens_avg=True).squeeze(['lat', 'lon']).load()  # flatten

seas5_ds_2 = seas5_data_reader.retrieve(var=seas5_var,
                                        lat=region_2['latmin'],
                                        lon=region_2['lonmin'],
                                        initmonths=initmonth,
                                        time=time_range,
                                        member=(0, 24), # We know that seas5 hindcast has members 0-24.
                                        ens_avg=True).squeeze(['lat', 'lon']).load()  # flatten

Taking Ensemble Average
Taking Ensemble Average


In [37]:
ufs_ds_1 = ufs_data_reader.retrieve(var=ufs_var,
                                    lat=region_1['latmin'],
                                    lon=region_1['lonmin'],
                                    initmonths=initmonth,
                                    time=time_range,
                                    ens_avg=True).squeeze(['lat', 'lon']).load()

ufs_ds_2 = ufs_data_reader.retrieve(var=ufs_var,
                                    lat=region_2['latmin'],
                                    lon=region_2['lonmin'],
                                    initmonths=initmonth,
                                    time=time_range,
                                    ens_avg=True).squeeze(['lat', 'lon']).load()

Taking Ensemble Average
Taking Ensemble Average


<h5>Calculate climatologies for each dataset (this may take a couple minutes)</h5>

In [38]:
ufs_stats_1 = stats.calc_climatology_anomaly(ufs_ds_1, area_mean=False, use_member_climatology=False)
ufs_stats_2 = stats.calc_climatology_anomaly(ufs_ds_2, area_mean=False, use_member_climatology=False)

In [39]:
seas5_stats_1 = stats.calc_climatology_anomaly(seas5_ds_1, area_mean=False, use_member_climatology=False)
seas5_stats_2 = stats.calc_climatology_anomaly(seas5_ds_2, area_mean=False, use_member_climatology=False)

</h5>Normalize the data. z = (X - mu) / sigma</h5>

In [40]:
# Normalize UFS datasets
ufs_da_1 = stats.normalize(da=ufs_ds_1[ufs_var], stats=ufs_stats_1)
ufs_da_2 = stats.normalize(da=ufs_ds_2[ufs_var], stats=ufs_stats_2)

In [41]:
# Normalize SEAS5 datasets
seas5_da_1 = stats.normalize(da=seas5_ds_1[seas5_var], stats=seas5_stats_1)
seas5_da_2 = stats.normalize(da=seas5_ds_2[seas5_var], stats=seas5_stats_2)

<h3>Calculate NAO Index</h3>

<h5>Take difference between 2 locations and store result into new datasets</h5>

In [43]:
ufs_ds_nao = (ufs_da_2 - ufs_da_1).to_dataset()
seas5_ds_nao = (seas5_da_2 - seas5_da_1).to_dataset()

<h2>Now that NAO Index is calculated, we can form Precip composites</h2>

<h5>First, classify which forecast leads are in positive or negative phase</h5>

In [44]:
# This is when SEAS5 NAO is positive or negative
seas5_positive_exclude_initleads = []
seas5_negative_exclude_initleads = []

for this_init in seas5_ds_nao.init.values:    
    for this_lead in seas5_ds_nao.lead.values:        
        # This NAO value
        this_nao_value = seas5_ds_nao[seas5_var].sel(init=this_init, lead=this_lead).values.item()
        
        # Is this NAO value non-positive or non-negative?
        if this_nao_value >= 0:
            seas5_negative_exclude_initleads.append((this_init, this_lead))
            
        elif this_nao_value <= 0:
            seas5_positive_exclude_initleads.append((this_init, this_lead))

In [45]:
# This is when UFS NAO is positive or negative
ufs_positive_exclude_initleads = []
ufs_negative_exclude_initleads = []

for this_init in ufs_ds_nao.init.values:    
    for this_lead in ufs_ds_nao.lead.values:        
        # This NAO value
        this_nao_value = ufs_ds_nao[ufs_var].sel(init=this_init, lead=this_lead).values.item()
        
        # Is this NAO value non-positive or non-negative?
        if this_nao_value >= 0:
            ufs_negative_exclude_initleads.append((this_init, this_lead))
            
        elif this_nao_value <= 0:
            ufs_positive_exclude_initleads.append((this_init, this_lead))

In [ ]:
------------

<h3>Read in datasets and do some preprocessing</h3>

In [ ]:
ufs_data_reader = dr.getDataReader(datasource='UFS',           
                                   experiment = ufs_experiment,
                                   # filename=f'experiments/phase_1/{ufs_model_1}/atm_monthly.zarr',                     
                                   model='atm')

In [ ]:
ufs_data_reader.dataset()

In [ ]:
# Our Regridder requires 1 UFS-type data reader and,
# if not another UFS model, a time-based model like ERA5.  

# What we will do is instantiate a UFS data reader,
# but then supplant its data with the SEAS5 dataset.
seas5_data_reader = dr.getDataReader(datasource='UFS',                  
                                     experiment = 'baseline',
                                     model='atm')

In [ ]:
# Collect all sst seas5 files in known directory.
seas5_file_list = os.listdir(seas5_dir)

# Prepend file paths.
seas5_file_list = [os.path.join(seas5_dir, this_file) for this_file in seas5_file_list]

In [ ]:
# netcdf4 package is needed here.
seas5_ds = xr.open_mfdataset(seas5_file_list, engine='netcdf4')

In [ ]:
# Rename dimensions/coordinates to match our init+lead paradigm.
seas5_ds = seas5_ds.rename_dims({'forecast_reference_time': 'init',
                                 'forecastMonth': 'lead',
                                  'number': 'member'})

seas5_ds = seas5_ds.rename({'forecast_reference_time': 'init',
                            'forecastMonth': 'lead',
                            'number': 'member'})

# Wrap it up into a DataReader object.
seas5_data_reader.update(ds=seas5_ds) 

In [ ]:
ufs_ds = ufs_data_reader.retrieve(var=ufs_var,
                                  time=time_range,
                                  initmonths=initmonth,
                                  lead=(min(leads), max(leads)),
                                  ens_avg=True)

In [ ]:
seas5_ds = seas5_data_reader.retrieve(var=seas5_var,
                                      time=time_range,
                                      initmonths=initmonth,
                                      lead=(min(leads), max(leads)),
                                      member=(0, 24), # We know seas5 hindcast has members 0-24
                                      ens_avg=True)

In [ ]:
# Scale fields.
ufs_ds = ufs_ds * ufs_scaling_factor                                                                          
seas5_ds = seas5_ds * seas5_scaling_factor

In [ ]:
# Update
ufs_data_reader.update(ds=ufs_ds)
seas5_data_reader.update(ds=seas5_ds)

In [ ]:
len(ufs_data_reader.dataset().lat.values)

In [ ]:
len(seas5_data_reader.dataset().lat.values)

In [ ]:
regridder = Regrid.Regrid(data_reader1=ufs_data_reader,                                                              
                          data_reader2=seas5_data_reader,                                                            
                          method='linear')

In [ ]:
# REGRID.
regridder.regrid(var=ufs_var)

In [ ]:
# Get Xarray dataset objects.
if regridder.highres_grid == 1:
    ufs_ds = regridder.regridded.dataset()
    seas5_ds = seas5_data_reader.dataset()
    
elif regridder.highres_grid == 2:
    ufs_ds = ufs_data_reader.dataset()
    seas5_ds = regridder.regridded.dataset()

In [ ]:
ufs_data_reader.update(ds=ufs_ds)
seas5_data_reader.update(ds=seas5_ds)

In [ ]:
ds1_elnino_statistics = stats.calc_composite_layers(data_reader=seas5_data_reader,
                                                    var=seas5_var,
                                                    statistics=['anomaly'],
                                                    exclude_initleads=elnino_exclude_initleads)

ds1_lanina_statistics = stats.calc_composite_layers(data_reader=seas5_data_reader,
                                                    var=seas5_var,
                                                    statistics=['anomaly'],
                                                    exclude_initleads=lanina_exclude_initleads)

ds2_elnino_statistics = stats.calc_composite_layers(data_reader=ufs_data_reader,
                                                    var=ufs_var,
                                                    statistics=['anomaly'],
                                                    exclude_initleads=elnino_exclude_initleads)

ds2_lanina_statistics = stats.calc_composite_layers(data_reader=ufs_data_reader,
                                                    var=ufs_var,
                                                    statistics=['anomaly'],
                                                    exclude_initleads=lanina_exclude_initleads)

<p>
---------------------------------------------------------------<br>
We now have 4 datasets of statistics:<br>
<br>
ds1_elnino_statistics<br>
ds1_lanina_statistics<br>
ds2_elnino_statistics<br>
ds2_lanina_statistics<br>
---------------------------------------------------------------
</p>

<h3>Run t-tests</h3>

In [ ]:
# Get position of init and lead axes
dims = list(ds1_elnino_statistics['anomaly'].dims) 
ttest_axes = []
if 'init' in dims:
    ttest_axes.append(dims.index("init"))
if 'lead' in dims:
    ttest_axes.append(dims.index("lead"))

In [ ]:
# EL NINO
anomaly_tstatistic_elnino, anomaly_pvalue_elnino =\
    scipy.stats.ttest_ind(a=ds1_elnino_statistics['anomaly'].values,
                          b=ds2_elnino_statistics['anomaly'].values,
                          axis=ttest_axes,
                          alternative='two-sided')

# LA NINA
anomaly_tstatistic_lanina, anomaly_pvalue_lanina =\
    scipy.stats.ttest_ind(a=ds1_lanina_statistics['anomaly'].values,
                          b=ds2_lanina_statistics['anomaly'].values,
                          axis=ttest_axes,
                          alternative='two-sided')

In [ ]:
p_values = xr.Dataset(
    data_vars={
        'anomaly_pvalue_elnino': (('lat', 'lon'), anomaly_pvalue_elnino),
        'anomaly_pvalue_lanina': (('lat', 'lon'), anomaly_pvalue_lanina)
        
    },
    coords={
        'lat': ufs_data_reader.dataset().lat.values,
        'lon': ufs_data_reader.dataset().lon.values
    }
)

<h1>Anomaly</h1>

In [ ]:
# This is the DataArray for the Composite
ds1_elnino_composite = ds1_elnino_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])
ds1_lanina_composite = ds1_lanina_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])

ds2_elnino_composite = ds2_elnino_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])
ds2_lanina_composite = ds2_lanina_statistics['anomaly'].sel(lead=list(leads)).mean(['init', 'lead'])

# Calculate the difference in anomalies
elnino_ds2_minus_ds1 = ds2_elnino_composite - ds1_elnino_composite
lanina_ds2_minus_ds1 = ds2_lanina_composite - ds1_lanina_composite

# Calculate Correlations
elnino_corr = xr.corr(ds1_elnino_composite, ds2_elnino_composite).values.item()
lanina_corr = xr.corr(ds1_lanina_composite, ds2_lanina_composite).values.item()

# Calculate Correlations over a smaller region
if region is not None:
    
    # latitudes are ordered 90 to -90
    lat_slice = slice(region['latmax'], region['latmin'])
    lon_slice = slice(region['lonmin'], region['lonmax'])
    
    elnino_corr_region = xr.corr(ds1_elnino_composite.sel(lat=lat_slice, lon=lon_slice),
                                 ds2_elnino_composite.sel(lat=lat_slice, lon=lon_slice)).values.item()
    
    lanina_corr_region = xr.corr(ds1_lanina_composite.sel(lat=lat_slice, lon=lon_slice),
                                 ds2_lanina_composite.sel(lat=lat_slice, lon=lon_slice)).values.item()

<h5>Make labels for the plots</h5>

In [ ]:
elnino_event_years = list(ds1_elnino_statistics.groupby('init.year').groups.keys())
lanina_event_years = list(ds1_lanina_statistics.groupby('init.year').groups.keys())

# Convert every year into 'YY for compactness.
elnino_event_years = [f"'{str(y)[2:4]}" for y in elnino_event_years]
lanina_event_years = [f"'{str(y)[2:4]}" for y in lanina_event_years]

elnino_event_years = f"Events: {' '.join([f'{y}' for y in elnino_event_years])}"  # convert to string
lanina_event_years = f"Events: {' '.join([f'{y}' for y in lanina_event_years])}"  # convert to string

# Label for init and leads
initlead_label = f"init {initmonth}\nlead {' '.join(filter(str.isdigit, str(leads)))}"

# Label for correlation
elnino_corr_label = f'corr: {elnino_corr:.2f}'
lanina_corr_label = f'corr: {lanina_corr:.2f}'

if region is not None:
    elnino_corr_label = f'{elnino_corr_label}\nregion: {elnino_corr_region:.2f}'
    lanina_corr_label = f'{lanina_corr_label}\nregion: {lanina_corr_region:.2f}'

<h3>Generate figures</h3>

In [ ]:
%%capture captured_output

# Instantiate buffers
buffer1 = BytesIO()
buffer2 = BytesIO()
buffer3 = BytesIO()
buffer4 = BytesIO()
buffer5 = BytesIO()
buffer6 = BytesIO()

# Make 6 plots
plot_kwargs = {'cmap_label': 'mm day-1',
               'cmap': 'BrBG',
               'topleft_label': initlead_label, 
               'region': region,
               'dpi': 200}

plot1 = stats.plot_composite(da = ds1_elnino_composite,
                             title=f'SEAS5 Precipitation Anomaly (El Nino)',
                             vmin=-6, vmax=6,
                             subtitle=elnino_event_years, **plot_kwargs)

plt.savefig(buffer1, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot2 = stats.plot_composite(da = ds1_lanina_composite,
                             title=f'SEAS5 Precipitation Anomaly (La Nina)',
                             vmin=-6, vmax=6,
                             subtitle=lanina_event_years, **plot_kwargs)

plt.savefig(buffer2, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot3 = stats.plot_composite(da = ds2_elnino_composite,
                             title=f'{ufs_data_reader.datasource} Precipitation Anomaly (El Nino)',
                             vmin=-6, vmax=6,
                             subtitle=elnino_event_years, **plot_kwargs)

plt.savefig(buffer3, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot4 = stats.plot_composite(da = ds2_lanina_composite,
                             title=f'{ufs_data_reader.datasource} Precipitation Anomaly (La Nina)',
                             vmin=-6, vmax=6,
                             subtitle=lanina_event_years, **plot_kwargs)

plt.savefig(buffer4, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot5 = stats.plot_composite(da = elnino_ds2_minus_ds1,
                             shading = p_values['anomaly_pvalue_elnino'],
                             shading_threshold = 0.05,
                             title=f'{ufs_data_reader.datasource} minus SEAS5 Precipitation Anomaly (El Nino)',
                             vmin=-4, vmax=4,
                             subtitle=elnino_event_years,
                             bottomright_label=elnino_corr_label,
                             **plot_kwargs)

plt.savefig(buffer5, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

plot6 = stats.plot_composite(da = lanina_ds2_minus_ds1,
                             shading = p_values['anomaly_pvalue_lanina'],
                             shading_threshold = 0.05,
                             title=f'{ufs_data_reader.datasource} minus SEAS5 Precipitation Anomaly (La Nina)',
                             vmin=-4, vmax=4,
                             subtitle=lanina_event_years,
                             bottomright_label=lanina_corr_label,
                             **plot_kwargs)

plt.savefig(buffer6, format='png', bbox_inches='tight')
# -----------------------------------------------------------------------------

<h3>Construct 3 row x 2 column multi-paneled image</h3>

In [ ]:
# Convert to images
image1 = Image.open(buffer1)
image2 = Image.open(buffer2)
image3 = Image.open(buffer3)
image4 = Image.open(buffer4)
image5 = Image.open(buffer5)
image6 = Image.open(buffer6)

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(11, 11), dpi=200)

axs[0, 0].imshow(image1)
axs[0, 0].axis('off')

axs[0, 1].imshow(image2)
axs[0, 1].axis('off')

axs[1, 0].imshow(image3)
axs[1, 0].axis('off')

axs[1, 1].imshow(image4)
axs[1, 1].axis('off')

axs[2, 0].imshow(image5)
axs[2, 0].axis('off')

axs[2, 1].imshow(image6)
axs[2, 1].axis('off')

plt.gca().set_frame_on(False)
plt.tight_layout(pad=.05)
plt.show()

In [ ]:
del buffer1, buffer2, buffer3, buffer4, buffer5, buffer6
del image1, image2, image3, image4, image5, image6
del plot1, plot2, plot3, plot4, plot5, plot6, plt
gc.collect()